# 2. Treinamento e avaliação
Compara os modelos candidatos por validação cruzada no treino, gera probabilidades OOF para selecionar o threshold maximizando F2 (β=2, priorizando recall), ajusta o modelo final e avalia uma única vez no teste reservado. Salva métricas, previsões OOF, análise de thresholds e gráficos.


In [1]:
import json
import joblib
import pandas as pd
from sklearn.metrics import f1_score, fbeta_score, precision_score, recall_score
from breast_cancer.config import (
    BEST_MODEL_DIR, CANDIDATE_MODEL_DIR, DATA_PATH, FIGURE_DIR,
    OOF_REPORT_DIR, REPORT_DIR, THRESHOLD_BETA,
)
from breast_cancer.data import load_data
from breast_cancer.preprocessing import prepare_data
from breast_cancer.models import make_random_forest, make_xgboost
from breast_cancer.validation import (
    cross_validated_probabilities, cross_validation_scores,
    select_threshold, threshold_analysis,
)
from breast_cancer.evaluation import evaluate_model
from breast_cancer.prediction import load_model_artifact, predict_proba
from breast_cancer.visualization import (
    save_confusion_matrix, save_pr_curve, save_roc_curve,
)

X_train, X_test, y_train, y_test = prepare_data(load_data(DATA_PATH))
print(f'Treino: {X_train.shape}; teste intocado: {X_test.shape}')


Treino: (341, 30); teste intocado: (228, 30)


In [2]:
xgboost = make_xgboost()
xgb_cv = cross_validation_scores(xgboost, X_train, y_train, cv=3, scoring='accuracy')
xgb_oof_probabilities = cross_validated_probabilities(xgboost, X_train, y_train, cv=5)
xgb_threshold = select_threshold(
    y_train, xgb_oof_probabilities, beta=THRESHOLD_BETA
)
xgb_oof_metrics = evaluate_model(y_train, xgb_oof_probabilities, xgb_threshold)
xgboost.fit(X_train, y_train)
xgb_test_probabilities = predict_proba(xgboost, X_test)
xgb_test_metrics = evaluate_model(y_test, xgb_test_probabilities, xgb_threshold)

print('XGBoost | CV accuracy (3 folds):', xgb_cv)
print(f'OOF | AUC-ROC: {xgb_oof_metrics["auc_roc"]:.3f}; threshold: {xgb_threshold:.3f}; F2: {fbeta_score(y_train, xgb_oof_metrics["predicted_labels"], beta=THRESHOLD_BETA):.3f}')
print(f'Teste reservado | AUC-ROC: {xgb_test_metrics["auc_roc"]:.3f}; AUPRC: {xgb_test_metrics["auprc"]:.3f}; accuracy: {xgb_test_metrics["accuracy"]:.3f}')
print(xgb_test_metrics['classification_report'])


XGBoost | CV accuracy (3 folds): [0.95614035 0.94736842 0.94690265]
OOF | AUC-ROC: 0.987; threshold: 0.444; F2: 0.951
Teste reservado | AUC-ROC: 0.997; AUPRC: 0.996; accuracy: 0.974
              precision    recall  f1-score   support

     Benigno      0.966     0.993     0.979       143
     Maligno      0.988     0.941     0.964        85

    accuracy                          0.974       228
   macro avg      0.977     0.967     0.972       228
weighted avg      0.974     0.974     0.974       228



In [3]:
forest = make_random_forest()
forest_cv = cross_validation_scores(forest, X_train, y_train, cv=3, scoring='accuracy')
forest.fit(X_train, y_train)
forest_probabilities = predict_proba(forest, X_test)
forest_metrics = evaluate_model(y_test, forest_probabilities, threshold=0.5)
print('Random Forest | CV accuracy (3 folds):', forest_cv)
print(f"AUC-ROC: {forest_metrics['auc_roc']:.3f}; AUPRC: {forest_metrics['auprc']:.3f}; accuracy: {forest_metrics['accuracy']:.3f}")
print(forest_metrics['classification_report'])


Random Forest | CV accuracy (3 folds): [0.95614035 0.94736842 0.92035398]
AUC-ROC: 0.994; AUPRC: 0.992; accuracy: 0.969
              precision    recall  f1-score   support

     Benigno      0.959     0.993     0.976       143
     Maligno      0.988     0.929     0.958        85

    accuracy                          0.969       228
   macro avg      0.973     0.961     0.967       228
weighted avg      0.970     0.969     0.969       228



In [4]:
candidate_cv = {'xgboost': xgb_cv, 'random_forest': forest_cv}
best_model_name = max(candidate_cv, key=lambda name: candidate_cv[name].mean())
candidate_models = {'xgboost': xgboost, 'random_forest': forest}
best_model = candidate_models[best_model_name]

if best_model_name == 'xgboost':
    best_oof_probabilities = xgb_oof_probabilities
    best_threshold = xgb_threshold
    best_oof_metrics = xgb_oof_metrics
else:
    best_oof_probabilities = cross_validated_probabilities(
        make_random_forest(), X_train, y_train, cv=5
    )
    best_threshold = select_threshold(
        y_train, best_oof_probabilities, beta=THRESHOLD_BETA
    )
    best_oof_metrics = evaluate_model(y_train, best_oof_probabilities, best_threshold)
    best_model.fit(X_train, y_train)

best_test_probabilities = predict_proba(best_model, X_test)
best_test_metrics = evaluate_model(y_test, best_test_probabilities, best_threshold)
best_model_path = BEST_MODEL_DIR / f'{best_model_name}_best.joblib'
CANDIDATE_MODEL_DIR.mkdir(parents=True, exist_ok=True)
BEST_MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
OOF_REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

joblib.dump(xgboost, CANDIDATE_MODEL_DIR / 'xgboost.joblib')
joblib.dump(forest, CANDIDATE_MODEL_DIR / 'random_forest.joblib')
joblib.dump({
    'model': best_model,
    'threshold': best_threshold,
    'feature_names': list(X_train.columns),
    'model_name': best_model_name,
}, best_model_path)

oof_predictions = pd.DataFrame({
    'row_index': X_train.index,
    'y_true': y_train.to_numpy(),
    'probability_malignant': best_oof_probabilities,
    'predicted_label': (best_oof_probabilities >= best_threshold).astype(int),
})
oof_predictions.to_csv(OOF_REPORT_DIR / 'oof_predictions.csv', index=False)
threshold_table = threshold_analysis(
    y_train, best_oof_probabilities, beta=THRESHOLD_BETA
)
threshold_table.to_csv(OOF_REPORT_DIR / 'threshold_analysis.csv', index=False)

save_roc_curve(
    best_test_metrics['fpr'], best_test_metrics['tpr'],
    best_test_metrics['auc_roc'], FIGURE_DIR / 'curva_roc.png',
)
save_pr_curve(
    best_test_metrics['precision_curve'], best_test_metrics['recall_curve'],
    best_test_metrics['auprc'], FIGURE_DIR / 'curva_pr.png',
)
save_confusion_matrix(
    y_test, best_test_metrics['predicted_labels'], FIGURE_DIR / 'matriz_confusao.png',
)

summary = {
    'best_model': best_model_name,
    'threshold': best_threshold,
    'selection_metric': 'mean_cv_accuracy',
    'threshold_selection_metric': 'oof_f_beta',
    'threshold_beta': THRESHOLD_BETA,
    'cv_accuracy': {name: scores.tolist() for name, scores in candidate_cv.items()},
    'oof': {
        'folds': 5,
        'beta': THRESHOLD_BETA,
        'auc_roc': best_oof_metrics['auc_roc'],
        'auprc': best_oof_metrics['auprc'],
        'precision_malignant': precision_score(y_train, best_oof_metrics['predicted_labels']),
        'recall_malignant': recall_score(y_train, best_oof_metrics['predicted_labels']),
        'f1_malignant': f1_score(y_train, best_oof_metrics['predicted_labels']),
        'f_beta_malignant': fbeta_score(y_train, best_oof_metrics['predicted_labels'], beta=THRESHOLD_BETA),
        'accuracy_at_selected_threshold': best_oof_metrics['accuracy'],
        'confusion_matrix': best_oof_metrics['confusion_matrix'].tolist(),
        'classification_report': best_oof_metrics['classification_report'],
    },
    'test': {
        'auc_roc': best_test_metrics['auc_roc'],
        'auprc': best_test_metrics['auprc'],
        'precision_malignant': precision_score(y_test, best_test_metrics['predicted_labels']),
        'recall_malignant': recall_score(y_test, best_test_metrics['predicted_labels']),
        'f1_malignant': f1_score(y_test, best_test_metrics['predicted_labels']),
        'f_beta_malignant': fbeta_score(y_test, best_test_metrics['predicted_labels'], beta=THRESHOLD_BETA),
        'accuracy': best_test_metrics['accuracy'],
        'confusion_matrix': best_test_metrics['confusion_matrix'].tolist(),
        'classification_report': best_test_metrics['classification_report'],
    },
    'candidates': {
        'xgboost': {'test_auc_roc': xgb_test_metrics['auc_roc'],
                    'test_auprc': xgb_test_metrics['auprc'],
                    'test_accuracy': xgb_test_metrics['accuracy']},
        'random_forest': {'test_auc_roc': forest_metrics['auc_roc'],
                          'test_auprc': forest_metrics['auprc'],
                          'test_accuracy': forest_metrics['accuracy']},
    },
}
(REPORT_DIR / 'metrics.json').write_text(
    json.dumps(summary, indent=2, ensure_ascii=False), encoding='utf-8'
)
print(f'Melhor modelo por CV: {best_model_name}')
print(f'Threshold OOF: {best_threshold:.4f}')
print(f'Teste reservado | AUC-ROC: {best_test_metrics["auc_roc"]:.4f}; AUPRC: {best_test_metrics["auprc"]:.4f}; recall: {recall_score(y_test, best_test_metrics["predicted_labels"]):.4f}; F1: {f1_score(y_test, best_test_metrics["predicted_labels"]):.4f}; F2: {fbeta_score(y_test, best_test_metrics["predicted_labels"], beta=THRESHOLD_BETA):.4f}')
print(f'Resultados salvos em {REPORT_DIR}; modelo final: {best_model_path}')


Melhor modelo por CV: xgboost
Threshold OOF: 0.4444
Teste reservado | AUC-ROC: 0.9972; AUPRC: 0.9961; recall: 0.9412; F1: 0.9639; F2: 0.9501
Resultados salvos em C:\Users\mateu_g1p9kd2\OneDrive\Documentos\GitHub\Predicao-cancer-de-mama-vs2\reports; modelo final: C:\Users\mateu_g1p9kd2\OneDrive\Documentos\GitHub\Predicao-cancer-de-mama-vs2\models\best_model\xgboost_best.joblib


## Inferência com o modelo selecionado
O artefato contém o modelo, o threshold escolhido por OOF e a ordem das features esperadas.


In [5]:
artifact = load_model_artifact(best_model_path)
predict_proba(
    artifact['model'], X_test, feature_names=artifact['feature_names']
)[:3]


array([0.99400926, 0.79233664, 0.01121518], dtype=float32)